# Pima Indians Diabetes Prediction & Imputation Pipeline
### Multi-Disease Prediction System using Machine Learning (Minor Project Part-I)
**Dataset:** Pima Indians Diabetes Database (768 records, 8 clinical features, Outcome 0/1)

---
### 1. Mathematical Formulation & Biological Reality
* In tabular clinical datasets, values of $0$ in physiological metrics like **Glucose, Blood Pressure, Skin Thickness, Insulin, and BMI** are biologically impossible in living patients.
* These must be audited, converted to `NaN`, and imputed using class-stratified medians:
  $$\hat{x}_{ij} = \text{Median}(\{x_{kj} \mid y_k = y_i, x_{kj} > 0\})$$
* **Random Forest Feature Importance (MDI):**
  $$I(X_j) = \frac{1}{T} \sum_{t=1}^{T} \sum_{v \in t, v(s)=X_j} \Delta I(v, t)$$


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.autolayout'] = True
%matplotlib inline

### 2. Data Loading & Biological Zero Anomaly Audit

In [ ]:
cols = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
data_path = os.path.join('Dataset minor', 'pima-indians-diabetes.csv')
df_raw = pd.read_csv(data_path, names=cols)
display(df_raw.head())

zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
zero_audit = {c: [(df_raw[c] == 0).sum(), (df_raw[c] == 0).mean() * 100] for c in zero_cols}
audit_df = pd.DataFrame(zero_audit, index=['Zero Count', 'Percentage (%)']).T
display(audit_df)

### 3. Class-Stratified Median Imputation

In [ ]:
df_clean = df_raw.copy()
df_clean[zero_cols] = df_clean[zero_cols].replace(0, np.nan)
for col in zero_cols:
    df_clean[col] = df_clean[col].fillna(df_clean.groupby('Outcome')[col].transform('median'))

print("Missing values after imputation:", df_clean.isnull().sum().sum())
display(df_clean.describe().T[['mean', 'std', 'min', '50%', 'max']])

### 4. Visualizations: Before vs After Imputation & Correlations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.kdeplot(df_raw['Insulin'], ax=axes[0], label='Raw (with zeros)', color='#e74c3c', lw=2)
sns.kdeplot(df_clean['Insulin'], ax=axes[0], label='Cleaned (Imputed)', color='#27ae60', lw=2.5)
axes[0].set_title("Insulin Distribution: Before vs After Imputation", fontweight='bold')
axes[0].legend()

sns.heatmap(df_clean.corr(), annot=True, fmt=".2f", cmap="YlGnBu", ax=axes[1], square=True)
axes[1].set_title("Pima Indians Correlation Matrix", fontweight='bold')
plt.show()

### 5. Train/Test Split & Benchmarking

In [ ]:
X = df_clean.drop('Outcome', axis=1)
y = df_clean['Outcome']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    'Support Vector Machine (RBF)': SVC(kernel='rbf', C=1.0, probability=True, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=7),
    'Decision Tree (CART)': DecisionTreeClassifier(max_depth=4, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.08, random_state=42)
}

results = {}
for name, model in models.items():
    if name in ['Logistic Regression', 'Support Vector Machine (RBF)', 'K-Nearest Neighbors']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
        
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall (Sensitivity)': recall_score(y_test, y_pred),
        'Specificity': tn / (tn + fp),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    }

display(pd.DataFrame(results).T.style.highlight_max(axis=0, color='#d4efdf'))

### 6. Random Forest Feature Importance

In [ ]:
rf = models['Random Forest']
fi = pd.DataFrame({'Feature': X.columns, 'Importance': rf.feature_importances_}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(9, 5))
plt.barh(fi['Feature'], fi['Importance'], color='#27ae60', edgecolor='black')
plt.title("Random Forest Feature Importance (Diabetes Biomarkers)", fontweight='bold')
plt.xlabel("Gini Importance")
plt.gca().invert_yaxis()
plt.show()